# Laya V3.0 — SlidingWindowFusion: search for the best sliding-attention replacement

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/Laya_V30_SlidingWindowFusion_Colab.ipynb)

### Architecture and scientific goal

**TARGET: held-out local output cosine ≥ 0.99** for every converted sliding-attention layer; measure teacher decision agreement, task accuracy, NMSE and CUDA latency on the full model, too. This is a **target, not a measured new result**.

In `transformers==4.57.6`, original ModernBERT sliding attention has a *bidirectional radius*: `original.local_attention=(R,R)`, typically **(64,64)** from `config.local_attention=128`. Its true receptive field is `2*R+1=129` (because distance ≤ R is inclusive). **Window size W129 is an exact native-radius reference**, *not* a new attention-free mechanism. The smaller W33, W65 and W97 windows remove context and must empirically recover quality.

### Replacement: RoPE-corrected SlidingWindowFusion

- Copy each layer's own pretrained **QKV, Wo, dropout and local RoPE**. QKV and RoPE stay frozen.
- Use genuine **block/halo sliding-window SDPA**, O(T·W) score memory. Never materialize global T×T teacher attention within the student.
- Add learned 7-, 33-, and 65-position depthwise cellular/convolution branches and per-head, per-token dynamic routing. Start very close to the exact local attention path.
- First **screen W33 / W65 / W97 / W129** at one representative sliding layer, compare blind probe and separate audit after 250 optimizer steps.
- Select the **fastest probe-qualified** window (cosine ≥0.99 *and* NMSE ≤0.025); if none passes, fall back to native-radius W129. Audit is *not* used for selecting the winning radius.
- Independently distill **every sliding layer**, with all layers' teacher I/O captured from only one teacher forward per cache batch, and retain the selected probe checkpoint.
- Measure cumulative quality each time a layer is replaced, then run the untouched official test and report CUDA full-model latency and a checkpoint reload check.
- Optionally install a previously trained V2.9 all-global checkpoint before sliding-layer training, to test a **both global+sliding converted** student. The checkpoint must already exist in the Colab runtime; this is disabled by default.

**Acceptance gate:** `all_layers_pass` checks every per-layer audit cosine ≥0.99 (plus NMSE ≤0.025). `quality_gate` checks held-out DEV decision agreement ≥0.99. The report prints PASS / NOT REACHED; failed layers remain visible and are not silently hidden. Teacher-window parity is expected by construction; a smaller-window speedup or 0.99 result is *not guaranteed*. Unlike global attention W1024, W129 is already small, and an extra convolution mixture might be slower than native fused SDPA; verify benchmark before claiming acceleration.



In [ ]:
#@title 1. Install
%pip -q install "transformers==4.57.6" "datasets>=2.20" "huggingface_hub>=0.25,<1" "safetensors>=0.4"
%pip -q install -U "git+https://github.com/vtavakkoli/TinyCeNN-LM.git@main"


In [ ]:
#@title 2. Teacher, candidate windows and dataset configuration
from pathlib import Path
import copy, json, math, random, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer
from transformers.models.modernbert.modeling_modernbert import apply_rotary_pos_emb
from tinycenn_lm.laya_lab.pdelta import PDelta3GDN2CLVRAttention
from tinycenn_lm.laya_lab.core import _depthwise_sequence_conv, _valid_tokens
from tinycenn_lm.standalone_decision import (
    QTYPES, build_checkpoint_model, build_sequence, collate_items,
    normalize_question,
)
from tinycenn_lm.decision_training import require_finite
from tinycenn_lm.sliding_window_fusion import SlidingWindowFusion, local_sliding_sdpa

SEED = 2028
SOURCE_MODEL = "convaiinnovations/laya-typed-decisions" #@param {type:"string"}
LAYER = 1  # first representative sliding layer, reassigned for all-layer loop
WINDOW_RADII = (16, 32, 48, 64) #@param {type:"raw"}
SCREEN_STEPS = 250 #@param {type:"integer"}
LAYER_STEPS = 300 #@param {type:"integer"}
BLOCK_SIZE = 64 #@param {type:"integer"}
LOCAL_NMSE_GATE = 0.025
DEV_AGREEMENT_GATE = 0.99
RUN_ALL_SLIDING = True #@param {type:"boolean"}
COMBINE_WITH_V29_GLOBAL = False #@param {type:"boolean"}
V29_CHECKPOINT = "/content/Laya_V29_AllGlobal_FastWindowMemory/v29_all_global_W1024.pt" #@param {type:"string"}

LANDMARK_STRIDE = 16 #@param {type:"integer"}
FEATURE_DIM = 96 #@param {type:"integer"}
TRAIN_CASES = 600 #@param {type:"integer"}
DEV_CASES = 180 #@param {type:"integer"}
MAX_LEN = 512 #@param {type:"integer"}
BATCH = 3 #@param {type:"integer"}
STEPS = SCREEN_STEPS #@param {type:"integer"}
TRAIN_CACHE_BATCHES = 24 #@param {type:"integer"}
PROBE_CACHE_BATCHES = 8 #@param {type:"integer"}
AUDIT_CACHE_BATCHES = 8 #@param {type:"integer"}
CHECK_EVERY = 50
TARGET_COSINE = 0.99
OUTPUT_DIR = Path("/content/Laya_V30_SlidingFusion")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert STEPS > 0 and LAYER_STEPS > 0
assert len(WINDOW_RADII) > 0 and all(isinstance(w, int) and w >= 1 for w in WINDOW_RADII)
assert LANDMARK_STRIDE > 0 and FEATURE_DIM > 0

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("This notebook needs a CUDA GPU (Colab > Runtime > Change runtime type).")
torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda")
amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False) else torch.float16

source_dir = Path(snapshot_download(
    SOURCE_MODEL,
    allow_patterns=["rl_agent_config.json", "model.safetensors", "encoder/*", "tokenizer/*"],
))
teacher, source_cfg = build_checkpoint_model(source_dir)
student_eval, _ = build_checkpoint_model(source_dir)
tokenizer = AutoTokenizer.from_pretrained(str(source_dir / "tokenizer"))
teacher.to(device).eval().requires_grad_(False)
student_eval.to(device).eval().requires_grad_(False)
teacher.encoder.config.reference_compile = False
student_eval.encoder.config.reference_compile = False

print("GPU:", torch.cuda.get_device_name(0), "| AMP:", amp_dtype)
GLOBAL_LAYERS = [i for i, layer in enumerate(teacher.encoder.layers)
                 if tuple(layer.attn.local_attention) == (-1, -1)]
SLIDING_LAYERS = [i for i, layer in enumerate(teacher.encoder.layers)
                  if tuple(layer.attn.local_attention) != (-1, -1)]
assert SLIDING_LAYERS and LAYER in SLIDING_LAYERS
NATIVE_RADIUS = teacher.encoder.layers[LAYER].attn.local_attention[0]
WINDOW_RADII = tuple(r for r in WINDOW_RADII if 0 < r <= NATIVE_RADIUS)
WINDOW_RADII = tuple(sorted(set(WINDOW_RADII + (NATIVE_RADIUS,))))
print("Original sliding radius:", NATIVE_RADIUS,
      "| true teacher window:", 2*NATIVE_RADIUS+1)
print("Candidate positions:", [2*r+1 for r in WINDOW_RADII])
print("Global layers untouched by default:", GLOBAL_LAYERS)
print("Sliding layers to replace:", SLIDING_LAYERS)
print("Teacher:", SOURCE_MODEL, "| official test split stays untouched")


In [ ]:
#@title 3. Leak-free train/dev data and teacher I/O helpers
def _jsonish(x):
    if isinstance(x, str):
        try:
            return json.loads(x)
        except Exception:
            return x
    return x

def _canon_label(x):
    s = str(x).strip().lower()
    try:
        v = float(s)
        if math.isfinite(v) and v.is_integer():
            return str(int(v))
    except Exception:
        pass
    return s

def _gold_metadata(q, gold_answer):
    if not isinstance(gold_answer, dict):
        raise ValueError("missing gold answer")
    probs_raw = _jsonish(gold_answer.get("probabilities", {}))
    if not isinstance(probs_raw, dict) or len(probs_raw) < 2:
        raise ValueError("gold probabilities missing or malformed")

    gold_keys = [str(k) for k in probs_raw.keys()]
    if q["t"] == "choice":
        labels = [str(k) for k in q["crit"].keys()]
    elif q["t"] == "noul":
        labels = ["false", "true"]
    else:
        labels = list(gold_keys)
        if all(_canon_label(k).lstrip("-").isdigit() for k in labels):
            labels = sorted(labels, key=lambda k: int(_canon_label(k)))

    lookup = {_canon_label(k): float(v) for k, v in probs_raw.items()}
    if len(labels) != len(gold_keys) or any(_canon_label(k) not in lookup for k in labels):
        raise ValueError("Gold labels do not match encoded option order")

    gold_probs = np.asarray([lookup[_canon_label(k)] for k in labels], dtype=np.float64)
    gold_probs = np.clip(gold_probs, 0.0, None)
    if not np.isfinite(gold_probs).all() or gold_probs.sum() <= 0:
        raise ValueError("invalid gold probability vector")
    gold_probs /= gold_probs.sum()

    gold_label = _canon_label(gold_answer.get("label", labels[int(gold_probs.argmax())]))
    canon_labels = [_canon_label(k) for k in labels]
    if gold_label not in canon_labels:
        raise ValueError(f"gold label {gold_label!r} not in {labels!r}")
    return labels, gold_probs.tolist(), canon_labels.index(gold_label)

def make_items_from_rows(rows, seed, shuffle_items):
    out = []
    head_max_len = int(source_cfg.get("head_max_len", 192))
    for row in rows:
        state = _jsonish(row["state"])
        questions = _jsonish(row["questions"])
        gold = _jsonish(row["gold"])
        case_id = str(row.get("id", ""))
        workflow = str(row.get("workflow", "unknown"))
        state_text = state if isinstance(state, str) else json.dumps(state, ensure_ascii=False)
        state_ids = tokenizer(
            state_text.replace(tokenizer.mask_token, " "),
            add_special_tokens=False,
        )["input_ids"]

        for qid, qdef in questions.items():
            q = normalize_question(qdef)
            ids, markers = build_sequence(
                tokenizer,
                state,
                q,
                max_len=min(MAX_LEN, int(source_cfg.get("max_len", MAX_LEN))),
                head_max_len=head_max_len,
                truncate_left=isinstance(state, list),
                state_ids=state_ids,
            )
            if len(markers) < 2:
                continue
            labels, gold_probs, gold_index = _gold_metadata(q, gold[qid])
            if len(labels) != len(markers):
                raise ValueError(
                    f"{case_id}/{qid}: {len(markers)} markers vs {len(labels)} gold labels"
                )
            out.append({
                "ids": ids,
                "markers": markers,
                "qtype": QTYPES[q["t"]],
                "qtype_name": q["t"],
                "qid": str(qid),
                "case_id": case_id,
                "workflow": workflow,
                "labels": labels,
                "gold_probs": gold_probs,
                "gold_index": int(gold_index),
            })
    if shuffle_items:
        random.Random(seed).shuffle(out)
    return out

rows = list(load_dataset("LocalLLaMA/typed-decisions", "all", split="train"))
rng = random.Random(SEED)
rng.shuffle(rows)
assert TRAIN_CASES + DEV_CASES <= len(rows)
train_rows = rows[:TRAIN_CASES]
dev_rows = rows[TRAIN_CASES:TRAIN_CASES + DEV_CASES]
train_items = make_items_from_rows(train_rows, SEED, True)
dev_items = make_items_from_rows(dev_rows, SEED + 1, True)

def direct_batch(selected):
    b = collate_items(selected, int(tokenizer.pad_token_id))
    return {k: v.to(device) for k, v in b.items()}

def batch(items, offset, n=BATCH):
    selected = [items[(offset + i) % len(items)] for i in range(n)]
    return direct_batch(selected)

def attn_io(model, layer_idx, b):
    layer = model.encoder.layers[layer_idx]
    got = {}
    def pre(mod, args, kwargs):
        got["x"] = args[0].detach()
        got["pos"] = None if kwargs.get("position_embeddings") is None else tuple(
            z.detach() for z in kwargs["position_embeddings"]
        )
        got["mask"] = None if kwargs.get("attention_mask") is None else kwargs["attention_mask"].detach()
        # ModernBERT SDPA requires position_ids and sliding_window_mask.
        # Replay the exact teacher call, including all positional/mask kwargs.
        got["call_kwargs"] = {
            key: (value.detach() if torch.is_tensor(value) else value)
            for key, value in kwargs.items()
        }
        if "position_ids" not in got["call_kwargs"] or "sliding_window_mask" not in got["call_kwargs"]:
            raise RuntimeError("Teacher attention hook did not capture ModernBERT SDPA kwargs")
    def post(mod, args, kwargs, out):
        got["y"] = out[0].detach()
    h1 = layer.attn.register_forward_pre_hook(pre, with_kwargs=True)
    h2 = layer.attn.register_forward_hook(post, with_kwargs=True)
    try:
        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=amp_dtype):
            model(**b)
    finally:
        h1.remove()
        h2.remove()
    return got

def cache_teacher_io(layer_idx, items, batches, offset=0):
    cache = []
    for i in range(batches):
        b = batch(items, offset + i * BATCH, BATCH)
        got = attn_io(teacher, layer_idx, b)
        cache.append({
            "x": got["x"],
            "pos": got["pos"],
            "mask": got["mask"],
            "call_kwargs": got["call_kwargs"],
            "y": got["y"],
            "valid": b["attention_mask"].bool(),
        })
    return cache

def local_metrics_from_cache(repl, cache):
    repl.eval()
    diff_sum = power_sum = cos_sum = 0.0
    tokens = 0
    with torch.no_grad():
        for entry in cache:
            with torch.autocast(device_type="cuda", dtype=amp_dtype):
                pred = repl(entry["x"], **entry["call_kwargs"])[0]
            require_finite(pred, "V3 sliding local probe")
            valid = entry["valid"]
            m = valid[:, :, None].float()
            d = pred.shape[-1]
            diff_sum += float((((pred.float() - entry["y"].float()) ** 2) * m).sum())
            power_sum += float(((entry["y"].float() ** 2) * m).sum())
            count = int(valid.sum())
            cos_sum += float(
                F.cosine_similarity(
                    pred.float()[valid],
                    entry["y"].float()[valid],
                    dim=-1,
                ).sum()
            )
            tokens += count
    nmse = diff_sum / max(power_sum, 1e-8)
    cosine = cos_sum / max(tokens, 1)
    score = nmse + 1.50 * (1.0 - cosine)
    return {"nmse": nmse, "cosine": cosine, "score": score, "tokens": tokens}

print("train/dev decisions:", len(train_items), len(dev_items))
assert len(dev_items) >= BATCH * (PROBE_CACHE_BATCHES + AUDIT_CACHE_BATCHES), "Increase DEV_CASES for disjoint probe and audit caches"
print("official test remains untouched; probe/audit I/O non-overlapping")


In [ ]:
#@title 4. Preflight: true native radius, exact RoPE/SDPA replay and gradient stability
teacher.encoder.config.reference_compile = False
entry_batch = batch(dev_items, 0, min(BATCH, 2))
entry = attn_io(teacher, LAYER, entry_batch)
assert entry["call_kwargs"]["sliding_window_mask"] is not None
assert entry["call_kwargs"]["position_ids"] is not None
with torch.no_grad(), torch.autocast("cuda", dtype=amp_dtype):
    replay = teacher.encoder.layers[LAYER].attn(
        entry["x"], **entry["call_kwargs"]
    )[0]
torch.testing.assert_close(replay.float(), entry["y"].float(),
                           atol=1e-3, rtol=1e-3)
print("Native sliding SDPA replay: PASS")
print("Mask:", tuple(entry["call_kwargs"]["sliding_window_mask"].shape),
      "| native radius:", teacher.encoder.layers[LAYER].attn.local_attention)
# Verify expected teacher-radius fidelity independently of optimization.
reference = SlidingWindowFusion(
    teacher.encoder.layers[LAYER].attn, radius=NATIVE_RADIUS,
    block_size=BLOCK_SIZE
).to(device).eval()
with torch.no_grad():
    reference.route_logits[:, 0].fill_(24.)
    reference.route_logits[:, 1:].fill_(-24.)
with torch.no_grad(), torch.autocast("cuda", dtype=amp_dtype):
    ref_output = reference(entry["x"], **entry["call_kwargs"])[0]
valid = entry_batch["attention_mask"].bool()
ref_cos = float(F.cosine_similarity(
    ref_output.float()[valid], entry["y"].float()[valid], -1
).mean())
ref_max_delta = float((ref_output.float()[valid] - entry["y"].float()[valid]).abs().max())
assert ref_cos >= 0.999, f"Teacher-sized kernel parity failed: cosine {ref_cos}"
print(f"R={NATIVE_RADIUS} packed sliding parity: cos={ref_cos:.6f}, "
      f"valid-token max Δ={ref_max_delta:.5g}")
for radius in WINDOW_RADII:
    model = SlidingWindowFusion(
        teacher.encoder.layers[LAYER].attn,
        radius=radius, block_size=BLOCK_SIZE
    ).to(device)
    model.train(); model.out_drop.eval()
    with torch.autocast("cuda", dtype=amp_dtype):
        prediction = model(entry["x"], **entry["call_kwargs"])[0]
        active = valid[:, :, None].float()
        loss = ((prediction.float()-entry["y"].float()).square()
                * active).sum() / (active.sum()*prediction.shape[-1]).clamp_min(1)
    grads = torch.autograd.grad(
        loss, [p for p in model.parameters() if p.requires_grad],
        allow_unused=True
    )
    assert any(g is not None and torch.isfinite(g).all()
               and bool(g.abs().sum()>0) for g in grads)
    print(f"W{2*radius+1}: finite gradients PASS, loss={float(loss.detach()):.6f}")
    del model, grads
torch.cuda.empty_cache()


In [ ]:
#@title 5. Fixed-budget training and checkpoint selection (probe only)
def score_local(m):
    return m["nmse"] + 1.50 * (1.0 - m["cosine"])

def optimizer_v28(model):
    core, output = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (output if name.startswith("Wo.") else core).append(p)
    params = core + output
    opt = torch.optim.AdamW([
        {"params": core, "lr": 0.003, "base_lr": 0.003},
        {"params": output, "lr": 0.00008, "base_lr": 0.00008},
    ], betas=(0.9, 0.95), weight_decay=2e-4)
    return opt, params

def train_400(model, train_cache, probe_cache, window):
    optimizer, params = optimizer_v28(model)
    scaler = torch.amp.GradScaler("cuda", enabled=amp_dtype == torch.float16, init_scale=128.0)
    best_metrics = local_metrics_from_cache(model, probe_cache)
    best_score = score_local(best_metrics)
    best_step = 0
    best_state = state_cpu(model)
    history = [{"step": 0, **best_metrics}]
    successes, attempts = 0, 0
    print(f"W{window} step    0/{STEPS}: NMSE={best_metrics['nmse']:.5f} cosine={best_metrics['cosine']:.5f}")

    while successes < STEPS:
        if attempts > STEPS + 200:
            raise RuntimeError("Too many AMP overflow/skipped optimizer steps")
        # A moderate cosine decay + short warmup is more stable than PDelta's
        # former 1e-2 opening learning rate, especially with trainable temperature.
        warm = min(1.0, (successes + 1) / 15.0)
        decay = 0.15 + 0.85 * 0.5 * (1.0 + math.cos(math.pi * successes / STEPS))
        for group in optimizer.param_groups:
            group["lr"] = group["base_lr"] * warm * decay

        model.train()
        model.out_drop.eval()
        entry = train_cache[attempts % len(train_cache)]
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", dtype=amp_dtype):
            output = model(entry["x"], **entry["call_kwargs"])[0]
            m = entry["valid"][:, :, None].float()
            target = entry["y"].float()
            pred = output.float()
            mse = (((pred - target) ** 2) * m).sum() / (m.sum() * pred.shape[-1]).clamp_min(1)
            power = ((target ** 2) * m).sum() / (m.sum() * pred.shape[-1]).clamp_min(1)
            nmse = mse / power.clamp_min(1e-8)
            cos = F.cosine_similarity(pred[entry["valid"]], target[entry["valid"]], dim=-1).mean()
            loss = nmse + 1.50 * (1.0 - cos)
        require_finite(loss, f"V28 W{window} training loss")
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        old_scale = scaler.get_scale()
        scaler.step(optimizer)
        scaler.update()
        attempts += 1
        if scaler.is_enabled() and scaler.get_scale() < old_scale:
            continue
        successes += 1
        if successes == 1 or successes % CHECK_EVERY == 0 or successes == STEPS:
            probe = local_metrics_from_cache(model, probe_cache)
            history.append({"step": successes, **probe})
            s = score_local(probe)
            if s < best_score:
                best_score, best_metrics, best_step = s, probe, successes
                best_state = state_cpu(model)
            print(
                f"W{window} step {successes:4d}/{STEPS}: "
                f"NMSE={probe['nmse']:.5f} cosine={probe['cosine']:.5f} "
                f"| best cosine={best_metrics['cosine']:.5f} @ {best_step}"
            )
    model.load_state_dict(best_state, strict=True)
    model.eval()
    return {
        "best_step": best_step, "best_probe": best_metrics,
        "attempts": attempts, "successful_updates": successes,
        "history": history,
    }


In [ ]:
#@title 7. End-to-end quality and CUDA latency diagnostics
@torch.no_grad()
def validate_pair(student, items=dev_items, limit=None):
    teacher.eval()
    student.eval()
    limit = len(items) if limit is None else min(int(limit), len(items))
    total = agree = teacher_gold = student_gold = 0
    js = []
    for off in range(0, limit, BATCH):
        chosen = items[off:min(off + BATCH, limit)]
        b = direct_batch(chosen)
        with torch.autocast(device_type="cuda", dtype=amp_dtype):
            tl, _ = teacher(**b)
            sl, _ = student(**b)
        require_finite(tl, "teacher validation logits")
        require_finite(sl, "SlidingFusion validation logits")
        allowed = b["marker_mask"].bool()
        tp = torch.softmax(tl.float().masked_fill(~allowed, -1e9), -1)
        sp = torch.softmax(sl.float().masked_fill(~allowed, -1e9), -1)
        gt = torch.tensor([row["gold_index"] for row in chosen], device=device)
        agree += int((tp.argmax(-1) == sp.argmax(-1)).sum())
        teacher_gold += int((tp.argmax(-1) == gt).sum())
        student_gold += int((sp.argmax(-1) == gt).sum())
        mid = 0.5 * (tp + sp)
        q = 0.5 * (tp * (tp.clamp_min(1e-8).log() - mid.clamp_min(1e-8).log())).sum(-1)
        q += 0.5 * (sp * (sp.clamp_min(1e-8).log() - mid.clamp_min(1e-8).log())).sum(-1)
        js.extend(q.cpu().tolist())
        total += len(chosen)
    return dict(
        decisions=total, agreement=agree / max(total, 1),
        mean_js=float(np.mean(js)),
        teacher_gold_accuracy=teacher_gold / max(total, 1),
        student_gold_accuracy=student_gold / max(total, 1),
    )

@torch.inference_mode()
def timed_attention(module, entry, warmup=5, repeats=15):
    module.eval()
    with torch.autocast(device_type="cuda", dtype=amp_dtype):
        for _ in range(warmup):
            module(entry["x"], **entry["call_kwargs"])
        torch.cuda.synchronize()
        ms = []
        for _ in range(repeats):
            start = time.perf_counter()
            module(entry["x"], **entry["call_kwargs"])
            torch.cuda.synchronize()
            ms.append((time.perf_counter() - start) * 1000)
    return float(np.median(ms))

def attention_coverage(cache, window):
    lengths = torch.cat([item["valid"].sum(-1).cpu() for item in cache]).numpy()
    # A centered W=96 attends only 47 tokens left and 48 right.
    # Length <= W alone does NOT guarantee all-to-all attention.
    full_context_limit = min((window - 1) // 2, window // 2) + 1
    return {
        "sequences": int(len(lengths)),
        "median_valid_tokens": float(np.median(lengths)),
        "max_valid_tokens": int(np.max(lengths)),
        "fraction_fully_covered_by_local_window": float(np.mean(lengths <= full_context_limit)),
        "full_context_length_limit": int(full_context_limit),
        "sequences_longer_than_window": int(np.sum(lengths > window)),
        "sequences_at_least_2x_window": int(np.sum(lengths >= 2 * window)),
    }



In [ ]:
#@title 7. Screen W33/W65/W97/W129; select smallest/fastest PASS on PROBE
DEV_QUALITY = dev_items[(PROBE_CACHE_BATCHES + AUDIT_CACHE_BATCHES)*BATCH:]
assert len(DEV_QUALITY) >= 120
screen_train = cache_teacher_io(LAYER, train_items, TRAIN_CACHE_BATCHES)
screen_probe = cache_teacher_io(LAYER, dev_items, PROBE_CACHE_BATCHES, offset=0)
screen_audit = cache_teacher_io(
    LAYER, dev_items, AUDIT_CACHE_BATCHES,
    offset=PROBE_CACHE_BATCHES * BATCH
)
screen_results = []
for radius in WINDOW_RADII:
    print("\n" + "="*85)
    print(f"Screen original sliding layer {LAYER}: W{2*radius+1} / native W{2*NATIVE_RADIUS+1}")
    print("="*85)
    random.seed(SEED); np.random.seed(SEED)
    torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
    model = SlidingWindowFusion(teacher.encoder.layers[LAYER].attn,
                                radius=radius, block_size=BLOCK_SIZE).to(device)
    first = local_metrics_from_cache(model, screen_probe)
    # Exact teacher-radius is a control; preserve the near-native start if it
    # already passes the 0.99 gate. Don't waste 250 steps increasing its drift.
    if radius == NATIVE_RADIUS and first["cosine"] >= TARGET_COSINE and first["nmse"] <= LOCAL_NMSE_GATE:
        info = {"best_step": 0, "best_probe": first,
                "successful_updates": 0,
                "history": [{"step": 0, **first}], "control": True}
        print("Native-radius reference passes at step 0; no training needed.")
    else:
        info = train_400(model, screen_train, screen_probe, 2*radius+1)
    audit = local_metrics_from_cache(model, screen_audit)
    teacher_ms = timed_attention(teacher.encoder.layers[LAYER].attn, screen_audit[0])
    student_ms = timed_attention(model, screen_audit[0])
    ckpt = OUTPUT_DIR / f"screen_layer{LAYER}_W{2*radius+1}.pt"
    torch.save({
        "version": "3.0", "radius": radius, "layer": LAYER,
        "state_dict": state_cpu(model), "training": info,
        "audit": audit, "config": model.config_dict(),
    }, ckpt)
    row = {
        "radius": radius, "window": 2*radius+1,
        "probe": info["best_probe"], "audit": audit,
        "steps": info["successful_updates"],
        "teacher_ms": teacher_ms, "student_ms": student_ms,
        "speedup": teacher_ms/max(student_ms,1e-9), "checkpoint": str(ckpt),
        "probe_pass": info["best_probe"]["cosine"] >= TARGET_COSINE
                      and info["best_probe"]["nmse"] <= LOCAL_NMSE_GATE,
        "control": radius == NATIVE_RADIUS,
    }
    screen_results.append(row)
    print(f"W{row['window']} PROBE cos={row['probe']['cosine']:.5f}, "
          f"AUDIT cos={audit['cosine']:.5f}, speedup={row['speedup']:.3f}x "
          f"| probe PASS={row['probe_pass']}")
    del model
    torch.cuda.empty_cache()
# Probe quality selects eligible candidates; latency measured only on the
# candidate itself, never use audit outputs for selection.
eligible = [r for r in screen_results if r["probe_pass"]]
if eligible:
    chosen = min(eligible, key=lambda r: (r["student_ms"], r["window"]))
else:
    chosen = next(r for r in screen_results if r["radius"] == NATIVE_RADIUS)
BEST_RADIUS = int(chosen["radius"])
print("\nBEST SCREEN BY PROBE-QUALITY + GPU LATENCY:", chosen)
print("WARNING: W129 is equivalent-context sliding attention, NOT attention-free.")


In [ ]:
#@title 8. Capture ALL sliding-layer teacher targets in one teacher pass per batch
# Reuse the same teacher forward for 20 different local attention targets.
# Keep the large I/O tensors on CPU FP16 to avoid T4 memory pressure.
def _cpu(x, compress=False):
    if x is None: return None
    y = x.detach().cpu()
    if compress and y.is_floating_point(): y = y.half()
    return y.contiguous()
def _gpu(x):
    return x.to(device, non_blocking=True) if torch.is_tensor(x) else x

def cache_all_sliding(items, batches, offset=0):
    out = {i: [] for i in SLIDING_LAYERS}
    got = {}
    hooks = []
    def make_pre(i):
        def pre(mod, args, kwargs):
            got[i] = {"x": args[0].detach(), "kwargs": kwargs}
        return pre
    def make_post(i):
        def post(mod, args, kwargs, response):
            got[i]["y"] = response[0].detach()
        return post
    for i in SLIDING_LAYERS:
        module = teacher.encoder.layers[i].attn
        hooks.append(module.register_forward_pre_hook(make_pre(i), with_kwargs=True))
        hooks.append(module.register_forward_hook(make_post(i), with_kwargs=True))
    try:
        for k in range(batches):
            b = batch(items, offset+k*BATCH)
            got.clear()
            with torch.no_grad(), torch.autocast("cuda", dtype=amp_dtype):
                teacher(**b)
            head = got[SLIDING_LAYERS[0]]
            kwargs = {key: _cpu(v) for key,v in head["kwargs"].items()}
            valid = _cpu(b["attention_mask"]).bool()
            for i in SLIDING_LAYERS:
                p = got[i]
                # All sliding layers have the same padded shape and position IDs.
                assert p["x"].shape[:2] == head["x"].shape[:2]
                out[i].append({
                    "x": _cpu(p["x"], compress=True),
                    "y": _cpu(p["y"], compress=True),
                    "call_kwargs": kwargs, "valid": valid
                })
            if (k+1)%4 == 0 or k+1 == batches:
                print(f"  cached {k+1}/{batches} batches for {len(SLIDING_LAYERS)} layers")
    finally:
        for hook in hooks: hook.remove()
    return out
def promote_cache(entries):
    return [{
        "x": _gpu(e["x"]), "y": _gpu(e["y"]),
        "valid": _gpu(e["valid"]),
        "call_kwargs": {k: _gpu(v) for k,v in e["call_kwargs"].items()}
    } for e in entries]

if RUN_ALL_SLIDING:
    train_cpu = cache_all_sliding(train_items, TRAIN_CACHE_BATCHES, offset=0)
    probe_cpu = cache_all_sliding(dev_items, PROBE_CACHE_BATCHES, offset=0)
    audit_cpu = cache_all_sliding(
        dev_items, AUDIT_CACHE_BATCHES, offset=PROBE_CACHE_BATCHES*BATCH
    )
    print("Shared all-layer teacher caches ready")
else:
    print("RUN_ALL_SLIDING=False, screening-only mode")


In [ ]:
#@title 9. Optional: combine with existing V2.9 ALL-GLOBAL pretrained conversion
combined_global_layers = []
if COMBINE_WITH_V29_GLOBAL:
    from tinycenn_lm.standalone_fastwindow_runtime import FastWindowAttention
    full_path = Path(V29_CHECKPOINT)
    if not full_path.is_file():
        raise FileNotFoundError(
            "V2.9 checkpoint not found. Copy it to V29_CHECKPOINT or disable COMBINE_WITH_V29_GLOBAL."
        )
    p = torch.load(full_path, map_location="cpu", weights_only=True)
    assert p["global_layers"] == GLOBAL_LAYERS
    assert "student_state" in p, "Need complete V2.9 checkpoint with student_state"
    for idx in GLOBAL_LAYERS:
        source = student_eval.encoder.layers[idx].attn
        student_eval.encoder.layers[idx].attn = FastWindowAttention(
            source, local_window=int(p["window"]), feature_dim=96,
            landmark_stride=16, max_landmarks=32
        ).to(device)
    student_eval.load_state_dict(p["student_state"], strict=True)
    student_eval.eval().requires_grad_(False)
    combined_global_layers = GLOBAL_LAYERS[:]
    print("Installed V2.9 full-attention conversion:", combined_global_layers)
else:
    print("Original global attention remains native (only sliding layers will change).")
baseline = validate_pair(student_eval, DEV_QUALITY, limit=120)
print("BEFORE ANY SLIDING CONVERSION:", baseline)


In [ ]:
#@title 10. Replace every sliding layer independently with selected window
ALL_LAYER_RESULTS = []
CUMULATIVE = []
LAYERS_CKPT_DIR = OUTPUT_DIR / f"layers_W{2*BEST_RADIUS+1}"
LAYERS_CKPT_DIR.mkdir(exist_ok=True)
if RUN_ALL_SLIDING:
    STEPS = LAYER_STEPS
    for idx in SLIDING_LAYERS:
        print("\n" + "="*90)
        print(f"TRAIN SLIDING LAYER {idx} / W{2*BEST_RADIUS+1} / native W{2*NATIVE_RADIUS+1}")
        print("="*90)
        tc = promote_cache(train_cpu[idx])
        pc = promote_cache(probe_cpu[idx])
        ac = promote_cache(audit_cpu[idx])
        original = teacher.encoder.layers[idx].attn
        model = SlidingWindowFusion(
            original, radius=BEST_RADIUS, block_size=BLOCK_SIZE
        ).to(device)
        current = local_metrics_from_cache(model, pc)
        checkpoint = LAYERS_CKPT_DIR / f"layer{idx:02d}.pt"
        if checkpoint.exists():
            state = torch.load(checkpoint, map_location="cpu", weights_only=True)
            assert state["radius"] == BEST_RADIUS and state["layer"] == idx
            model.load_state_dict(state["state_dict"], strict=True)
            info, audit = state["train"], state["audit"]
            print("Restored trained layer:", checkpoint)
        else:
            if current["cosine"] >= TARGET_COSINE and current["nmse"] <= LOCAL_NMSE_GATE:
                # Keep exact-quality initialization; no needless optimization.
                info = {"best_step": 0, "best_probe": current,
                        "successful_updates": 0, "history": [{"step":0, **current}]}
            else:
                info = train_400(model, tc, pc, 2*BEST_RADIUS+1)
            audit = local_metrics_from_cache(model, ac)
            torch.save({
                "version": "3.0", "layer": idx, "radius": BEST_RADIUS,
                "state_dict": state_cpu(model), "train": info, "audit": audit,
                "config": model.config_dict(),
            }, checkpoint)
        model.eval().requires_grad_(False)
        student_eval.encoder.layers[idx].attn = model
        metric = validate_pair(student_eval, DEV_QUALITY, limit=120)
        result = {
            "layer": idx, "radius": BEST_RADIUS,
            "probe": info["best_probe"], "audit": audit,
            "steps": info["successful_updates"],
            "passed": audit["cosine"] >= TARGET_COSINE
                      and audit["nmse"] <= LOCAL_NMSE_GATE,
            "checkpoint": str(checkpoint),
        }
        ALL_LAYER_RESULTS.append(result)
        CUMULATIVE.append({"through_layer": idx, "quality": metric})
        print(f"AUDIT cosine={audit['cosine']:.5f}, NMSE={audit['nmse']:.5f}, "
              f"PASS={result['passed']}")
        print(f"CUMULATIVE agreement={metric['agreement']:.5f}, "
              f"gold={metric['student_gold_accuracy']:.5f}")
        del tc,pc,ac,model
        train_cpu.pop(idx,None)
        probe_cpu.pop(idx,None)
        audit_cpu.pop(idx,None)
        torch.cuda.empty_cache()
    assert all(isinstance(student_eval.encoder.layers[i].attn, SlidingWindowFusion)
               for i in SLIDING_LAYERS)
    print("REPLACED ALL SLIDING LAYERS:", SLIDING_LAYERS)
else:
    print("Screening-only: no full-model conversion claimed.")


In [ ]:
#@title 11. Final blind test, full-model CUDA latency, acceptance gates and checkpoint
if not RUN_ALL_SLIDING:
    print("Skip full-model official test until RUN_ALL_SLIDING=True.")
else:
    pretest = OUTPUT_DIR / f"v30_all_sliding_W{2*BEST_RADIUS+1}_pretest.pt"
    torch.save({
        "version":"3.0", "selected_radius": BEST_RADIUS,
        "sliding_layers": SLIDING_LAYERS,
        "converted_global_layers": combined_global_layers,
        "student_state": state_cpu(student_eval),
        "local_results": ALL_LAYER_RESULTS, "cumulative": CUMULATIVE,
    }, pretest)
    print("SAVED FULL MODEL before official test:", pretest)
    test_rows = list(load_dataset("LocalLLaMA/typed-decisions", "all", split="test"))
    test_items = make_items_from_rows(test_rows, SEED+99, False)
    dev_final = validate_pair(student_eval, DEV_QUALITY)
    test_final = validate_pair(student_eval, test_items)
    @torch.no_grad()
    def time_model(m, rows, warmup=5, repeats=15):
        b = direct_batch(rows)
        m.eval()
        with torch.autocast("cuda",dtype=amp_dtype):
            for _ in range(warmup): m(**b)
            torch.cuda.synchronize()
            times=[]
            for _ in range(repeats):
                start=time.perf_counter(); m(**b); torch.cuda.synchronize()
                times.append(1000*(time.perf_counter()-start))
        return {"median_ms":float(np.median(times)),
                "p95_ms":float(np.percentile(times,95)),
                "padded_length":int(b["input_ids"].shape[1])}
    latency=[]
    for size in (1,BATCH):
        chosen=DEV_QUALITY[:size]
        t=time_model(teacher,chosen)
        s=time_model(student_eval,chosen)
        latency.append({"batch_size":size,"teacher":t,"student":s,
                        "speedup":t["median_ms"]/max(s["median_ms"],1e-8)})
    all_pass=all(row["passed"] for row in ALL_LAYER_RESULTS)
    e2e_pass=dev_final["agreement"] >= DEV_AGREEMENT_GATE
    report={
        "version":"3.0", "architecture":"SlidingWindowFusion",
        "target_cosine":TARGET_COSINE,"target_nmse":LOCAL_NMSE_GATE,
        "screening":screen_results,"selected_radius":BEST_RADIUS,
        "window_positions":2*BEST_RADIUS+1,
        "native_radius":NATIVE_RADIUS,"native_window_positions":2*NATIVE_RADIUS+1,
        "all_sliding_layers":SLIDING_LAYERS,
        "optional_converted_global_layers":combined_global_layers,
        "layer_results":ALL_LAYER_RESULTS,"cumulative":CUMULATIVE,
        "all_layer_gate_pass":all_pass,
        "dev_quality_gate_pass":e2e_pass,
        "dev_final":dev_final,"official_test_final":test_final,
        "official_test_used_for_selection":False,
        "latency":latency,
        "uses_attention":True, "dense_global_attention_in_replacement":False,
    }
    report_file=OUTPUT_DIR / f"v30_all_sliding_W{2*BEST_RADIUS+1}_report.json"
    report_file.write_text(json.dumps(report,indent=2,allow_nan=False))
    print("ALL LAYER COSINE GATE:", "PASS" if all_pass else "NOT REACHED")
    print("FULL MODEL DEV 0.99 AGREEMENT GATE:", "PASS" if e2e_pass else "NOT REACHED")
    print("DEV:",json.dumps(dev_final,indent=2))
    print("OFFICIAL TEST:",json.dumps(test_final,indent=2))
    print("LATENCY:",json.dumps(latency,indent=2))
    print("REPORT:", report_file)


In [ ]:
#@title 12. Reload exact converted model and check prediction parity
if RUN_ALL_SLIDING:
    payload=torch.load(pretest,map_location="cpu",weights_only=True)
    restored,_=build_checkpoint_model(source_dir)
    restored.to(device).eval().requires_grad_(False)
    restored.encoder.config.reference_compile=False
    if combined_global_layers:
        from tinycenn_lm.standalone_fastwindow_runtime import FastWindowAttention
        old=torch.load(Path(V29_CHECKPOINT),map_location="cpu",weights_only=True)
        for idx in combined_global_layers:
            original=restored.encoder.layers[idx].attn
            restored.encoder.layers[idx].attn=FastWindowAttention(
                original,local_window=int(old["window"]),
                feature_dim=96,landmark_stride=16,max_landmarks=32
            ).to(device)
    for idx in SLIDING_LAYERS:
        restored.encoder.layers[idx].attn=SlidingWindowFusion(
            restored.encoder.layers[idx].attn,radius=BEST_RADIUS,
            block_size=BLOCK_SIZE
        ).to(device)
    restored.load_state_dict(payload["student_state"],strict=True)
    restored.eval().requires_grad_(False)
    with torch.no_grad(),torch.autocast("cuda",dtype=amp_dtype):
        sample=direct_batch(DEV_QUALITY[:min(3,len(DEV_QUALITY))])
        current=student_eval(**sample)[0]
        loaded=restored(**sample)[0]
    torch.testing.assert_close(loaded.float(),current.float(),atol=1e-3,rtol=1e-3)
    print("RELOAD PARITY PASS: exact full-model logits.")


In [ ]:
#@title 13. Visualize attention architecture and audit comparison
import matplotlib.pyplot as plt
fig, axs=plt.subplots(1,2,figsize=(13,4.2))
axs[0].plot([x["window"] for x in screen_results],
            [x["probe"]["cosine"] for x in screen_results],
            marker="o",label="Probe cosine")
axs[0].plot([x["window"] for x in screen_results],
            [x["audit"]["cosine"] for x in screen_results],
            marker="x",ls="--",label="Audit cosine")
axs[0].axhline(.99,ls=":",color="gray",label="0.99 goal")
axs[0].set(xlabel="Local positions (2r+1)",ylabel="Mean valid-token cosine",
           title="Sliding window fidelity")
axs[0].legend(); axs[0].grid(alpha=.25)
axs[1].bar([str(x["window"]) for x in screen_results],
           [x["speedup"] for x in screen_results])
axs[1].axhline(1,ls="--",color="gray")
axs[1].set(xlabel="Window positions",ylabel="Teacher / student latency",
           title="Measured layer speedup (screening)")
axs[1].grid(axis="y",alpha=.25)
fig.tight_layout()
plot_file=OUTPUT_DIR / "v30_sliding_window_screen.png"
fig.savefig(plot_file,dpi=170,bbox_inches="tight")
plt.show()
if RUN_ALL_SLIDING:
    fig,ax=plt.subplots(figsize=(11,4))
    ax.plot([x["through_layer"] for x in CUMULATIVE],
            [x["quality"]["agreement"] for x in CUMULATIVE],
            marker="o",label="Dev teacher agreement")
    ax.axhline(.99,ls="--",color="gray",label="0.99 target")
    ax.set(xlabel="Last installed sliding layer",ylabel="Agreement",ylim=(0,1.02),
           title="Cumulative downstream quality after sliding conversion")
    ax.grid(alpha=.2);ax.legend();fig.tight_layout()
    cumulative_plot=OUTPUT_DIR/"v30_cumulative_sliding.png"
    fig.savefig(cumulative_plot,dpi=170,bbox_inches="tight")
    plt.show()
    print("PLOTS:",plot_file,cumulative_plot)
else:
    print("SCREEN PLOT:",plot_file)
